# Train TrOCR on IAM Handwriting Dataset

This notebook fine-tunes the Microsoft TrOCR model on the IAM Handwriting dataset using Hugging Face datasets (`Teklia/IAM-line`).

### ⚠️ IMPORTANT: Enable GPU ⚠️
Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU NOT Detected! Please change runtime type to GPU.")
    # raise RuntimeError("No GPU found. Training will be too slow.")

In [ ]:
# OPTIONAL: Mount Google Drive to save model checkpoints safely
# This prevents losing your model if Colab runtime disconnects!
from google.colab import drive
try:
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully.")
except:
    print("⚠️ Google Drive not mounted. Model will only be saved locally in Colab.")

In [ ]:
# 1. Clone or Update the repository
import os

if os.path.exists('handwriting_recog'):
    %cd handwriting_recog
    !git pull origin main
else:
    !git clone https://github.com/Bhuvan-018/handwriting_recog
    %cd handwriting_recog

In [ ]:
# 2. Install dependencies
!pip install -r requirements.txt

In [ ]:
# 3. Run the training script
# This will take approximately 1-2 hours on T4 GPU
!python train_hf.py

In [ ]:
# 4. Backup Model to Google Drive (Highly Recommended)
import os
import shutil
from datetime import datetime

MODEL_DIR = "models/trocr_finetuned_iam_hf"
DRIVE_PATH = "/content/drive/MyDrive/Handwriting_Model_Backup"

if os.path.exists(MODEL_DIR):
    # Create backup directory in Drive
    if os.path.exists("/content/drive/MyDrive"):
        os.makedirs(DRIVE_PATH, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M")
        zip_name = f"trocr_model_{timestamp}.zip"
        
        print(f"📦 Zipping model to {zip_name}...")
        !zip -r {zip_name} {MODEL_DIR}
        
        print(f"💾 Copying to Google Drive: {DRIVE_PATH}...")
        shutil.copy(zip_name, os.path.join(DRIVE_PATH, zip_name))
        print("✅ Backup successful! Check your Google Drive.")
    else:
        print("⚠️ Google Drive not mounted. Skipping backup.")
else:
    print("❌ Model directory not found. Did training complete successfully?")

In [ ]:
# 5. Deploy to Hugging Face Space
# This deploys the newly trained model directly to your Space

import os
import shutil

# --- Configuration ---
HF_TOKEN = "YOUR_HF_WRITE_TOKEN" # @param {type:"string"}
SPACE_ID = "bhuvan-018/handwriting-recognition" # @param {type:"string"}

# Clean inputs
HF_TOKEN = HF_TOKEN.strip()
SPACE_ID = SPACE_ID.strip()

MODEL_DIR = "models/trocr_finetuned_iam_hf"

# --- Deploy Logic ---
if not os.path.exists(MODEL_DIR):
    print(f"❌ Error: Model directory {MODEL_DIR} not found. Please run training first.")
else:
    # --- Reduce Size: Remove Checkpoints ---
    print("🧹 Cleaning up intermediate checkpoints to save space...")
    checkpoints = [d for d in os.listdir(MODEL_DIR) if d.startswith('checkpoint-')]
    for ckpt in checkpoints:
        ckpt_path = os.path.join(MODEL_DIR, ckpt)
        print(f"Removing {ckpt_path}...")
        shutil.rmtree(ckpt_path)

    # --- Deploy to Spaces ---
    if HF_TOKEN == "YOUR_HF_WRITE_TOKEN" or not HF_TOKEN:
        print("⚠️ Please enter your Hugging Face Write Token above!")
    else:
        print("🚀 Starting deployment to Hugging Face Space...")
        
        # Clean up any existing space_repo to prevent nesting issues
        if os.path.exists("space_repo"):
            shutil.rmtree("space_repo")
        
        # Install Git LFS
        !git lfs install
        
        # Configure Git
        !git config --global user.email "colab@example.com"
        !git config --global user.name "Colab User"
        
        # Clone your Hugging Face Space
        repo_url = f"https://{HF_TOKEN}@huggingface.co/spaces/{SPACE_ID}"
        !git clone {repo_url} space_repo
        
        # Check if app_gradio.py exists in CURRENT directory
        if not os.path.exists("app_gradio.py"):
             print("⚠️ app_gradio.py not found locally. Downloading from repo...")
             !wget https://raw.githubusercontent.com/Bhuvan-018/handwriting_recog/main/app_gradio.py
             !wget https://raw.githubusercontent.com/Bhuvan-018/handwriting_recog/main/requirements.txt
             !mkdir -p utils
             !wget -P utils https://raw.githubusercontent.com/Bhuvan-018/handwriting_recog/main/utils/preprocessing.py

        # Copy model files to the Space repo
        print(f"📦 Moving model files from {MODEL_DIR}...")
        !mkdir -p space_repo/models/trocr_finetuned_iam_hf
        
        # COPY files (Keep original for backup) 
        # Since we are on a fresh runtime with plenty of space, copy is safer than mv
        !cp -r {MODEL_DIR}/* space_repo/models/trocr_finetuned_iam_hf/
        
        # Copy app files (ensure they are up to date from the repo)
        print("📄 Copying app files...")
        !cp app_gradio.py space_repo/app.py
        !cp requirements.txt space_repo/
        if os.path.exists("utils"):
             !cp -r utils space_repo/
        
        # Commit and Push
        print("⬆️ Pushing to Hugging Face (this may take a few minutes)...")
        # Change directory to space_repo ONLY for git operations
        os.chdir("space_repo")
        
        # Set remote url again just to be safe
        !git remote set-url origin {repo_url}
        
        !git lfs track "*.bin"
        !git lfs track "*.safetensors"
        !git add .
        !git commit -m "Deploy fine-tuned model from Colab"
        
        # Use robust push
        !git push
        print("✅ Successfully deployed to Hugging Face Space!")
        # Go back to parent directory
        os.chdir("..")